Getting the rows with order=1(main questions)

In [2]:
import pandas as pd
data_path = r"C:\Users\rajur\Downloads\skill_builder_data_corrected_collapsed.csv"
df = pd.read_csv(data_path,encoding="cp1252",low_memory=False)
main_df=df.loc[df["original"]==1].copy()
print("Shape of main_df:", main_df.shape)
display(main_df.head())
# print(main_df["original"].value_counts())

Shape of main_df: (275458, 31)


,Unnamed: 0,order_id,assignment_id,user_id,assistment_id,problem_id,original,correct,attempt_count,ms_first_response,...,hint_count,hint_total,overlap_time,template_id,answer_id,answer_text,first_action,bottom_hint,opportunity,opportunity_original
0,1,33022537,277618,64525,33139,51424,1,1,1,32454,...,0,3,32454,30799,NaN,26,0,NaN,1,1.0
1,2,33022709,277618,64525,33150,51435,1,1,1,4922,...,0,3,4922,30799,NaN,55,0,NaN,2,2.0
2,3,35450204,220674,70363,33159,51444,1,0,2,25390,...,0,3,42000,30799,NaN,88,0,NaN,1,1.0
3,4,35450295,220674,70363,33110,51395,1,1,1,4859,...,0,3,4859,30059,NaN,41,0,NaN,2,2.0
4,5,35450311,220674,70363,33196,51481,1,0,14,19813,...,3,4,124564,30060,NaN,65,0,0.0,3,3.0


Parsing skill_id into Python sets.

In [3]:
def parse_skill_id(value):
    if pd.isna(value):
        return set()
    return set(value.split("_"))
main_df["skill_set"]=main_df["skill_id"].apply(parse_skill_id)

print(main_df[["skill_id", "skill_set"]].head(10))
print("\nMissing skill IDs:",
      main_df["skill_set"].apply(len).eq(0).sum())
# Verify consistency between original and parsed skill columns
missing_original = main_df["skill_id"].isna()
empty_parsed = main_df["skill_set"].apply(len).eq(0)

print("Missing original skill IDs:", missing_original.sum())
print("Empty parsed skill sets:", empty_parsed.sum())

print("Parsing consistency:",
      missing_original.equals(empty_parsed))


  skill_id skill_set
0     1_13   {13, 1}
1     1_13   {13, 1}
2     1_13   {13, 1}
3     1_13   {13, 1}
4     1_15   {15, 1}
5     1_15   {15, 1}
6     1_15   {15, 1}
7     1_13   {13, 1}
8     1_15   {15, 1}
9     1_13   {13, 1}

Missing skill IDs: 16059
Missing original skill IDs: 16059
Empty parsed skill sets: 16059
Parsing consistency: True


Establish chronological order in our working dataset within student

In [4]:

# Arrange each student's interactions in chronological order
main_df = (
    main_df.sort_values(["user_id", "order_id"])
           .reset_index(drop=True)
)

print("Shape:", main_df.shape)
print("\nFirst 10 rows:")
print(main_df[["user_id", "order_id", "problem_id"]].head(10))

Shape: (275458, 32)

First 10 rows:
   user_id  order_id  problem_id
0       14  21617623       93383
1       14  21617632       93407
2       14  21617641       93400
3       14  21617650       93419
4       14  21617659       93420
5       14  21617667       93415
6       14  21617675       93423
7       14  21617692       57695
8       14  21617731       58596
9       14  21617749       57647


Select columns required for our initial feature engineering

In [13]:

required_columns = [
    "user_id",
    "order_id",
    "problem_id",
    "correct",
    "skill_id",
    "skill_set",
    "attempt_count"
]

clean_df = main_df[required_columns].copy()

print("Clean dataset shape:", clean_df.shape)
print("\nColumns:", clean_df.columns.tolist())

Clean dataset shape: (275458, 7)

Columns: ['user_id', 'order_id', 'problem_id', 'correct', 'skill_id', 'skill_set', 'attempt_count']


## Verification of cleaned data set

In [14]:
# Basic validation of the cleaned dataset

print("Dataset shape:", clean_df.shape)

print("\nMissing values:")
print(clean_df.isna().sum())

print("\nUnique students:", clean_df["user_id"].nunique())
print("Unique problems:", clean_df["problem_id"].nunique())
print("Unique order IDs:", clean_df["order_id"].nunique())

print("\nCorrect values:")
print(clean_df["correct"].value_counts().sort_index())

Dataset shape: (275458, 7)

Missing values:
user_id              0
order_id             0
problem_id           0
correct              0
skill_id         16059
skill_set            0
attempt_count        0
dtype: int64

Unique students: 4217
Unique problems: 18209
Unique order IDs: 275458

Correct values:
correct
0     94105
1    181353
Name: count, dtype: int64


In [15]:
# Verify that order IDs increase within each student
is_sorted = (
    clean_df.groupby("user_id")["order_id"]
    .apply(lambda s: s.is_monotonic_increasing)
)

print("All students chronologically sorted:", is_sorted.all())
print("Students checked:", len(is_sorted))

All students chronologically sorted: True
Students checked: 4217


In [16]:
from pathlib import Path
processed_dir=Path("../data/processed")
processed_dir.mkdir(parents=True,exist_ok=True)

output_path=processed_dir/"main_questions_cleaned.csv"
clean_df.to_csv(output_path,index=False)
print("saved to:",output_path)
print("Rows saved:",len(clean_df))

saved to: ..\data\processed\main_questions_cleaned.csv
Rows saved: 275458
